# 04 — Compare the models

The full analysis has already been executed. By default this notebook displays its saved results. Set RUN_TRAINING to True to repeat the complete calculation; this overwrites derived results for the same frozen protocol and may take several minutes.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/model_analysis.py').is_file())
sys.path.insert(0,str(ROOT/'src'))


Five outer stratified folds estimate held-out performance. Three inner folds select settings using positive-class F1. Imputation, one-hot encoding and scaling are inside each model pipeline. LR/KNN numerical features are scaled; LR/tree/forest class weights are balanced. KNN vote weighting is separate from class balancing. No synthetic oversampling is used. The threshold stays at 0.5 for tuning and testing.

In [ ]:
from model_analysis import run, freeze
RUN_TRAINING = False
if RUN_TRAINING:
    run()
    from report_results import run as refresh_reports
    refresh_reports()
else:
    print("Displaying the completed run; no training requested in this cell.")
protocol = json.loads((ROOT/"results/protocol.json").read_text())
display(protocol)

E1 uses area, elevation and type. E2 removes type, a blueprint refinement. E3 adds climate on identical lakes/folds. E4 adds coordinates and basin; E1 provides the removal comparison. Separately, three grouped outer/inner folds hold out whole climate cells for E1 and E3; this does not establish transfer to a distant basin.

In [ ]:
display(pd.read_csv(ROOT/"results/fold_class_counts.csv"))
display(pd.read_csv(ROOT/"results/comparison_with_fold_variability.csv"))

Pooled F1 uses all outer predictions together. Mean-fold F1 averages five or three separate fold scores, so the numbers need not agree. Fold standard deviation is descriptive, not an independent confidence interval. The cells below replay every stored outer estimator and recalculate metrics.

In [ ]:
from verify_results import verify
verification = verify()

Do not use these held-out results to enlarge tuning grids, choose a new threshold or select a new feature set while claiming the same confirmatory evaluation. Further design changes require a clearly separate exploratory experiment.